## Imports

In [1]:
import dadi
import os
import pandas as pd
import vcf
import numpy as np
import shutil
import subprocess

In [2]:
%pwd

'/n/holylfs05/LABS/hopkins_lab/Users/pfmckenzie/projects/mybdadi'

## Define the pops

In [3]:
sympops = ["656A",
"667",
"669",
"680A",
"761",
"764",
"768D",
"769",
"786",
"794",
"797",
"798D",
"800",
"802",
"807",
"817",
"831",]

allopops = ["812",
"690",
"691",
"692",
"702",
"703",
"704",
"706",
"784",
"785",
"787",
"676",
"677",
"777",
"778",
"780",
"830",]

## Read in the metadata, filter down to P. drummondii samples only

In [4]:
metadata_full=pd.read_csv('./ddRAD_popgen_metadata.csv')
metadata = metadata_full[metadata_full.species.eq('drum')]

## Isolate the sympatric vs. allopatric data

In [5]:
symdata = metadata[[i in sympops for i in metadata.Population]]
allodata = metadata[[i in allopops for i in metadata.Population]]

In [6]:
# save a list of sympatric IDs
symids = list(symdata.Sample_ID)
# save a list of allopatric IDs
alloids = list(allodata.Sample_ID)

### total number in metadata that we could keep

In [9]:
len(symids) + len(alloids)

382

### get the list of samples we don't want to include

In [10]:
trim_subset = symids+alloids
exclude_samps = list(set(metadata_full.Sample_ID).difference(trim_subset))
exclude_samps.extend(['656A-25-1B','687-6','666-6-1','678-3-1','651-8-1','640-6-1','640-2-1','647-5-1']) # weird/hybrid samples

In [12]:
len(exclude_samps)

524

### get the names of samples in our full VCF

In [13]:
vcf_path = "/n/holylfs05/LABS/hopkins_lab/Users/pfmckenzie/projects/mybdadi/dadi_aws.vcf"

# Open the VCF file
vcf_reader = vcf.Reader(filename=vcf_path)

# Get the sample names
sample_names = vcf_reader.samples

len(sample_names)

722

### Remove any samples from the vcf that we don't want

In [14]:
[sample_names.remove(i) if i in sample_names else "not present" for i in exclude_samps]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 None,
 None,
 'not present',
 'not present',
 None,
 None,
 'not present',
 None,
 'not present',
 None,
 'not present',
 None,
 None,
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 None,
 None,
 None,
 None,
 'not present',
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 None,
 'not present',
 None,
 None,
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'not present',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None

In [15]:
# number remaining
len(sample_names)

305

### Subsample the VCF to only include these

In [16]:
def subsample_vcf(input_vcf, output_vcf, sample_list, snps_only=True):
    """
    subsamples VCF based on a list of samples and optionally keeps only SNPs.
    
    :param input_vcf: path to the input VCF file
    :param output_vcf: path to the output VCF file
    :param sample_list: list of sample IDs to keep
    :param snps_only: bool indicating whether to keep only SNPs
    """
    
    # form command
    samples = ",".join(sample_list)
    cmd1 = ["bcftools", "view", "-s", samples, "-o", "temp.vcf", input_vcf]
    
    # run
    subprocess.run(cmd1, check=True)
    
    if snps_only:
        # keep only the snps
        cmd2 = ["bcftools", "view", "-v", "snps", "-o", output_vcf, "temp.vcf"]
        
        # fun
        subprocess.run(cmd2, check=True)
        
        # delete temp
        subprocess.run(["rm", "temp.vcf"], check=True)
    else:
        # rename output
        subprocess.run(["mv", "temp.vcf", output_vcf], check=True)

# no singletons, strand-flipped, polarized vcf
vcf_path =  "/n/holylfs05/LABS/hopkins_lab/Users/pfmckenzie/projects/mybdadi/dadi_aws.vcf"

# sample them and write out vcf
subsample_vcf(vcf_path, './dadi_out_subsamp.vcf', sample_names)

### check our output

In [17]:
# specify the new vcf path
vcf_path = './dadi_out_subsamp.vcf'

# Open the VCF file
vcf_reader = vcf.Reader(filename=vcf_path)

# Get the sample names
sample_names = vcf_reader.samples

len(sample_names)

305

### match up sample ids with population ids

In [18]:
# Open the metadata for the samples
ddrad_metadata = pd.read_csv('./ddRAD_popgen_metadata.csv')

# Match each sample to a species
#species_assignments = [ddrad_metadata[ddrad_metadata.Sample_ID.eq(name)].species.iloc[0] for name in sample_names]
species_assignments = []
for name in sample_names:
    if name in symids:
        species_assignments.append("sym")
    elif name in alloids:
        species_assignments.append("allo")

In [75]:
# Make a pop file that matches each individual to a population
indiv_species_pairs = ['\t'.join(i) for i in zip(sample_names,species_assignments)]

ind2pop_path = './ind2pop_dadi.txt'
with open(ind2pop_path,'w') as f:
    for pair in indiv_species_pairs:
        print(pair, file=f)

### Count numbers of each pop, make the data dictionary

In [76]:
np.sum(np.array(species_assignments) == 'sym')

175

In [77]:
np.sum(np.array(species_assignments) == 'allo')

130

In [78]:
dd = dadi.Misc.make_data_dict_vcf(vcf_path, ind2pop_path,)

In [79]:
# two chroms per indiv - this is the max value that we could project from for sym
175*2

350

# Finding projection that maximizes segregating sites

### course-grained search (big steps)

In [ ]:
maxnum = 0
for symnum in range(305,350,8):
    for allonum in range(1,260,8):
        fs = dadi.Spectrum.from_data_dict(dd, ['sym','allo'], projections = [symnum,allonum], polarized = False)
        print([symnum,allonum])
        total_segregating = fs.S()
        print(total_segregating)
        if total_segregating > maxnum:
            maxnum = total_segregating
            bestcomb = [symnum,allonum]
print(f'best combo is {bestcomb} with {maxnum} segregating sites')

In [85]:
bestcomb

[81, 73]

### fine-grained search

In [88]:
maxnum = 0
for symnum in range(70,90,1):
    for allonum in range(60,80,1):
        fs = dadi.Spectrum.from_data_dict(dd, ['sym','allo'], projections = [symnum,allonum], polarized = False)
        print([symnum,allonum])
        total_segregating = fs.S()
        print(total_segregating)
        if total_segregating > maxnum:
            maxnum = total_segregating
            bestcomb = [symnum,allonum]
print(f'best combo is {bestcomb} with {maxnum} segregating sites')

[70, 60]
3912.3332811061105
[70, 61]
3906.8929699386777
[70, 62]
3916.936640485267
[70, 63]
3904.364962074238
[70, 64]
3914.1387154877557
[70, 65]
3904.3859620259705
[70, 66]
3913.932103988637
[70, 67]
3909.5025153214456
[70, 68]
3918.8302079738887
[70, 69]
3907.4826099179654
[70, 70]
3916.5837258861216
[70, 71]
3895.2741023566164
[70, 72]
3904.144044074349
[70, 73]
3888.343362532829
[70, 74]
3896.970739992415
[70, 75]
3878.434895068992
[70, 76]
3886.860548786516
[70, 77]
3868.4069193375126
[70, 78]
3876.6417608177767
[70, 79]
3856.4923490592596
[71, 60]
3907.9665996139065
[71, 61]
3903.4229907890904
[71, 62]
3913.382674748483
[71, 63]
3903.708413301038
[71, 64]
3913.418161174018
[71, 65]
3903.5693878160982
[71, 66]
3913.0551085643106
[71, 67]
3909.548643467189
[71, 68]
3918.8194475327555
[71, 69]
3907.3589002565495
[71, 70]
3916.406708985917
[71, 71]
3894.9803442123857
[71, 72]
3903.800490012868
[71, 73]
3890.9382953161294
[71, 74]
3899.5187736810094
[71, 75]
3881.899692047699
[71, 76

### best combo is sym=80,allo=70

# Next step: Set up files for dadi runs

### init directory

In [3]:
dadi_dir = './runs/'

In [4]:
runs_dirname = dadi_dir#os.path.join(dadi_dir,"cluster_runs")

In [91]:
if not os.path.exists(runs_dirname):
    os.mkdir(runs_dirname)

### point to portik paths

In [93]:
# source file path
source_file_path = '../ddrad_demography/dadi_pipeline/Optimize_Functions.py'

# copy the file
shutil.copy(source_file_path, runs_dirname)

'./runs/Optimize_Functions.py'

In [94]:
# source file path
source_file_path = '../ddrad_demography/dadi_pipeline/Two_Population_Pipeline/Models_2D.py'

# copy the file
shutil.copy(source_file_path, runs_dirname)

'./runs/Models_2D.py'

### move into directory

In [5]:
%cd {runs_dirname}

/n/holylfs05/LABS/hopkins_lab/Users/pfmckenzie/projects/mybdadi/runs


### general function to submit sbatch job on cluster

In [6]:
import subprocess

def submit_job_via_sbatch(script_path):
    """
    submit a job to slurm via sbatch

    :param script_path: path to script
    """
    try:
        # run sbatch command
        result = subprocess.run(['sbatch', script_path], capture_output=True, text=True)
        
        # if successful, print output
        if result.returncode == 0:
            print(f"Job submitted successfully:\n{result.stdout}")
        else:
            print(f"An error occurred while submitting the job:\n{result.stderr}")
    
    except Exception as e:
        print(f"An error occurred while executing the sbatch command: {e}")

# Running dadi

### params

In [128]:
model_name = 'asym_mig'#'split_nomig' # string -- model name in portik dadi pipeline
species_ordered = ['sym','allo'] # order matters here - see portik model illustrations
#reps = [1] # iterable, list of integer ids
proj = [80,70] # three projection values
pts = [150,215,280] # three grid sizes
vcf_path='/n/holylfs05/LABS/hopkins_lab/Users/pfmckenzie/projects/mybdadi/dadi_out_subsamp.vcf'

repnum=10

### slurm script text

In [129]:
slurm_script = """#!/bin/bash

#SBATCH --job-name=dadicpu       # Job name
#SBATCH --output=dadi_output_%j.log  # out and error log filename (%j is jobid)
#SBATCH --cpus-per-task 2
#SBATCH --ntasks=1
#SBATCH --time=49:59:00         # walltime HH:MM:SS
#SBATCH --mem-per-cpu=1G
#SBATCH -p sapphire # partition

# echo the start date and time to the standard out
echo "Start time:" $(date)

#module load cuda
#nvcc --version

python dadi_script_{runid}.py

# echo the start date and time to the standard out
echo "End time:" $(date)
"""

### dadi script

In [130]:
species_prefix = ''.join([i[0] for i in species_ordered])
proj_prefix = '-'.join([str(i) for i in proj])
pts_prefix = '-'.join([str(i) for i in pts])

pyscript = """import dadi
import os
import pandas as pd
import vcf
import numpy as np
import shutil

dadi.cuda_enabled(True)

# no singletons, strand-flipped, polarized vcf
vcf_path = "{vcf_path}"

# Open the VCF file
vcf_reader = vcf.Reader(filename=vcf_path)

# Get the sample names
sample_names = vcf_reader.samples

ind2pop_path = '../ind2pop_dadi.txt'

dd = dadi.Misc.make_data_dict_vcf(vcf_path, 
                                  ind2pop_path,)

proj = {proj}
species_ordered = {species_ordered}
fs = dadi.Spectrum.from_data_dict(dd, species_ordered, projections = proj, polarized = False)
fs.S()

###

import Optimize_Functions
import Models_2D

model_name = "{model_name}"

#make sure to define your extrapolation grid size (based on your projections)
# Higher than the largest projection size
# guidance here: https://groups.google.com/g/dadi-user/c/2hSq_Tjicso/m/7Ygr8TghDAAJ
pts = {pts}

#species_prefix = ''.join([i[0] for i in species_ordered])
#proj_prefix = '-'.join(proj)
#pts_prefix = '-'.join(pts)

#create a prefix based on the population names to label the output files
prefix = "{species_prefix}_{proj_prefix}_{pts_prefix}_{model_name}_{repnum}"

#**************
#Set the number of rounds here
rounds = 4

#define the lists for optional arguments
#you can change these to alter the settings of the optimization routine
reps = [10,20,30,40]
maxiters = [3,5,10,15]
folds = [3,2,2,1]

##Optimize_Functions.Optimize_Routine(fs, pts, prefix, "sym_mig", sym_mig, 3, 4, fs_folded=True, param_labels = p_labels, in_upper = upper, in_lower = lower)


#**************
#Indicate whether your frequency spectrum object is folded (True) or unfolded (False)
fs_folded = True

if model_name=="no_mig":
    # Split into two populations, no migration.
    Optimize_Functions.Optimize_Routine(fs, pts, prefix, "no_mig", Models_2D.no_mig, rounds, 3, fs_folded=fs_folded,
                                            reps=reps, maxiters=maxiters, folds=folds, param_labels = "nu1, nu2, T")

if model_name=="sym_mig":
    # Split into two populations, with continuous symmetric migration.
    Optimize_Functions.Optimize_Routine(fs, pts, prefix, "sym_mig", Models_2D.sym_mig, rounds, 4, fs_folded=fs_folded,
                                            reps=reps, maxiters=maxiters, folds=folds, param_labels = "nu1, nu2, m, T")

if model_name=="asym_mig":
    # Split into two populations, with continuous asymmetric migration.
    Optimize_Functions.Optimize_Routine(fs, pts, prefix, "asym_mig", Models_2D.asym_mig, rounds, 5, fs_folded=fs_folded,
                                            reps=reps, maxiters=maxiters, folds=folds, param_labels = "nu1, nu2, m12, m21, T")

""".format(repnum=repnum,
           proj=proj,
           pts=pts,
           model_name=model_name,
           species_ordered=species_ordered,
           species_prefix=species_prefix,
           pts_prefix=pts_prefix,
           proj_prefix=proj_prefix,
           vcf_path=vcf_path
          )

### write out scripts

In [131]:
runid = f"{species_prefix}_{proj_prefix}_{pts_prefix}_{model_name}_{repnum}"
with open(f"dadi_script_{runid}.py","w") as f:
    f.write(pyscript)
with open("run_dadi.slurm","w") as f:
    f.write(slurm_script.format(runid=runid))

### submit job

In [132]:
# Example usage:
script_path = "run_dadi.slurm"
submit_job_via_sbatch(script_path)

Job submitted successfully:
Submitted batch job 35659479

